# Titanic competition with TensorFlow Decision Forests

This notebook will take you through the steps needed to train a baseline Gradient Boosted Trees Model using TensorFlow Decision Forests and creating a submission on the Titanic competition. 

This notebook shows:

1. How to do some basic pre-processing. For example, the passenger names will be tokenized, and ticket names will be splitted in parts.
1. How to train a Gradient Boosted Trees (GBT) with default parameters
1. How to train a GBT with improved default parameters
1. How to tune the parameters of a GBTs
1. How to train and ensemble many GBTs

# Imports dependencies

In [1]:
import numpy as np
import pandas as pd
import os

import tensorflow as tf
import tensorflow_decision_forests as tfdf

print(f"Found TF-DF {tfdf.__version__}")

Found TF-DF 1.2.0


# Load dataset

In [2]:
train_df = pd.read_csv("/kaggle/input/titanic/train.csv")
serving_df = pd.read_csv("/kaggle/input/titanic/test.csv")

train_df.head(10)

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S
5,6,0,3,"Moran, Mr. James",male,NaN,0,0,330877,8.4583,NaN,Q
6,7,0,1,"McCarthy, Mr. Timothy J",male,54.0,0,0,17463,51.8625,E46,S
7,8,0,3,"Palsson, Master. Gosta Leonard",male,2.0,3,1,349909,21.0750,NaN,S
8,9,1,3,"Johnson, Mrs. Oscar W (Elisabeth Vilhelmina Berg)",female,27.0,0,2,347742,11.1333,NaN,S
9,10,1,2,"Nasser, Mrs. Nicholas (Adele Achem)",female,14.0,1,0,237736,30.0708,NaN,C


# Prepare dataset

We will apply the following transformations on the dataset.

1. Tokenize the names. For example, "Braund, Mr. Owen Harris" will become ["Braund", "Mr.", "Owen", "Harris"].
2. Extract any prefix in the ticket. For example ticket "STON/O2. 3101282" will become "STON/O2." and 3101282.

In [3]:
def preprocess(df):
    df = df.copy()
    
    def normalize_name(x):
        return " ".join([v.strip(",()[].\"'") for v in x.split(" ")])
    
    def ticket_number(x):
        return x.split(" ")[-1]
        
    def ticket_item(x):
        items = x.split(" ")
        if len(items) == 1:
            return "NONE"
        return "_".join(items[0:-1])
    
    def get_title(name):
        title = name.split(",")[1].split(".")[0].strip()
        rare_titles = ["Lady","Countess","Capt","Col","Don","Dr",
                       "Major","Rev","Sir","Jonkheer","Dona"]
        if title in rare_titles:
            return "Rare"
        mapping = {"Mlle":"Miss","Ms":"Miss","Mme":"Mrs"}
        return mapping.get(title, title)
    
    df["Title"] = df["Name"].apply(get_title)
    df["FamilySize"] = df["SibSp"] + df["Parch"] + 1
    df["Deck"] = df["Cabin"].astype(str).str[0]
    df["FarePerPerson"] = df["Fare"] / df["FamilySize"]
    
    # ここではAge埋めをしない(後段で train の中央値をまとめて適用するため)
    
    df["Name"] = df["Name"].apply(normalize_name)
    df["Ticket_number"] = df["Ticket"].apply(ticket_number)
    df["Ticket_item"] = df["Ticket"].apply(ticket_item)
    return df


def fill_age_with_train_medians(df, age_medians_by_title, global_median):
    df = df.copy()
    df["Age"] = df.apply(
        lambda row: age_medians_by_title.get(row["Title"], global_median)
                    if pd.isna(row["Age"]) else row["Age"],
        axis=1
    )
    return df


# ① ベース前処理(Title, FamilySize, Deck, FarePerPersonまで。Ageはまだ欠損あり)
preprocessed_train_df = preprocess(train_df)
preprocessed_serving_df = preprocess(serving_df)

# ② trainだけを使って Title ごとの中央値を計算
age_medians_by_title = preprocessed_train_df.groupby("Title")["Age"].median()
global_median = preprocessed_train_df["Age"].median()  # 万一testにしかないTitleが来た場合の保険

# ③ train・test両方に、trainの中央値を適用
preprocessed_train_df = fill_age_with_train_medians(preprocessed_train_df, age_medians_by_title, global_median)
preprocessed_serving_df = fill_age_with_train_medians(preprocessed_serving_df, age_medians_by_title, global_median)

# ④ 確認
print(preprocessed_train_df["Age"].isna().sum())    # 0になっているはず
print(preprocessed_serving_df["Age"].isna().sum())  # 0になっているはず

preprocessed_train_df.head(5)

0
0


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,Title,FamilySize,Deck,FarePerPerson,Ticket_number,Ticket_item
0,1,0,3,Braund Mr Owen Harris,male,22.0,1,0,A/5 21171,7.2500,NaN,S,Mr,2,n,3.62500,21171,A/5
1,2,1,1,Cumings Mrs John Bradley Florence Briggs Thayer,female,38.0,1,0,PC 17599,71.2833,C85,C,Mrs,2,C,35.64165,17599,PC
2,3,1,3,Heikkinen Miss Laina,female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S,Miss,1,n,7.92500,3101282,STON/O2.
3,4,1,1,Futrelle Mrs Jacques Heath Lily May Peel,female,35.0,1,0,113803,53.1000,C123,S,Mrs,2,C,26.55000,113803,NONE
4,5,0,3,Allen Mr William Henry,male,35.0,0,0,373450,8.0500,NaN,S,Mr,1,n,8.05000,373450,NONE


Let's keep the list of the input features of the model. Notably, we don't want to train our model on the "PassengerId" and "Ticket" features.

In [4]:
input_features = list(preprocessed_train_df.columns)
input_features.remove("Ticket")
input_features.remove("PassengerId")
input_features.remove("Survived")
#input_features.remove("Ticket_number")
input_features.remove("Ticket_item")  # 負の重要度だったため除外
input_features.remove("Embarked")     # 負の重要度だったため除外
input_features.remove("Cabin")  # Deckが情報を吸収済みのため除外

print(f"Input features: {input_features}")

Input features: ['Pclass', 'Name', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Title', 'FamilySize', 'Deck', 'FarePerPerson', 'Ticket_number']


# Convert Pandas dataset to TensorFlow Dataset

In [5]:
def tokenize_names(features, labels=None):
    """Divite the names into tokens. TF-DF can consume text tokens natively."""
    features["Name"] =  tf.strings.split(features["Name"])
    return features, labels

train_ds = tfdf.keras.pd_dataframe_to_tf_dataset(preprocessed_train_df,label="Survived").map(tokenize_names)
serving_ds = tfdf.keras.pd_dataframe_to_tf_dataset(preprocessed_serving_df).map(tokenize_names)

# Train model with default parameters

### Train model

First, we are training a GradientBoostedTreesModel model with the default parameters.

In [6]:
model = tfdf.keras.GradientBoostedTreesModel(
    verbose=0, # Very few logs
    features=[tfdf.keras.FeatureUsage(name=n) for n in input_features],
    exclude_non_specified_features=True, # Only use the features in "features"
    random_seed=1234,
)
model.fit(train_ds)

self_evaluation = model.make_inspector().evaluation()
print(f"Accuracy: {self_evaluation.accuracy} Loss:{self_evaluation.loss}")

[INFO 2026-07-05T05:08:00.346896447+00:00 kernel.cc:1214] Loading model from path /tmp/tmpt1qpgtp5/model/ with prefix 91bdfd2b565248a8
[INFO 2026-07-05T05:08:00.356465356+00:00 abstract_model.cc:1311] Engine "GradientBoostedTreesQuickScorerExtended" built
[INFO 2026-07-05T05:08:00.356542787+00:00 kernel.cc:1046] Use fast generic engine


Please report this to the TensorFlow team. When filing the bug, set the verbosity to 10 (on Linux, `export AUTOGRAPH_VERBOSITY=10`) and attach the full output.
Cause: could not get source code
To silence this warning, decorate the function with @tf.autograph.experimental.do_not_convert
Accuracy: 0.804347813129425 Loss:0.8727316856384277


# Train model with improved default parameters

Now you'll use some specific parameters when creating the GBT model

In [7]:
model = tfdf.keras.GradientBoostedTreesModel(
    verbose=0, # Very few logs
    features=[tfdf.keras.FeatureUsage(name=n) for n in input_features],
    exclude_non_specified_features=True, # Only use the features in "features"
    
    #num_trees=2000,
    
    # Only for GBT.
    # A bit slower, but great to understand the model.
    # compute_permutation_variable_importance=True,
    
    # Change the default hyper-parameters
    # hyperparameter_template="benchmark_rank1@v1",
    
    #num_trees=1000,
    #tuner=tuner
    
    min_examples=1,
    categorical_algorithm="RANDOM",
    #max_depth=4,
    shrinkage=0.05,
    #num_candidate_attributes_ratio=0.2,
    split_axis="SPARSE_OBLIQUE",
    sparse_oblique_normalization="MIN_MAX",
    sparse_oblique_num_projections_exponent=2.0,
    num_trees=2000,
    #validation_ratio=0.0,
    random_seed=1234,
    
)
model.fit(train_ds)

self_evaluation = model.make_inspector().evaluation()
print(f"Accuracy: {self_evaluation.accuracy} Loss:{self_evaluation.loss}")

[INFO 2026-07-05T05:08:04.114954576+00:00 kernel.cc:1214] Loading model from path /tmp/tmpp8ay4_ep/model/ with prefix 71eaa96edd484b13
[INFO 2026-07-05T05:08:04.12803705+00:00 decision_forest.cc:661] Model loaded with 40 root(s), 2224 node(s), and 12 input feature(s).
[INFO 2026-07-05T05:08:04.128095547+00:00 kernel.cc:1046] Use fast generic engine


Accuracy: 0.8260869383811951 Loss:0.8949137330055237


Let's look at the model and you can also notice the information about variable importance that the model figured out

In [8]:
model.summary()

Model: "gradient_boosted_trees_model_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
Total params: 1
Trainable params: 0
Non-trainable params: 1
_________________________________________________________________
Type: "GRADIENT_BOOSTED_TREES"
Task: CLASSIFICATION
Label: "__LABEL"

Input Features (12):
	Age
	Deck
	FamilySize
	Fare
	FarePerPerson
	Name
	Parch
	Pclass
	Sex
	SibSp
	Ticket_number
	Title

No weights

Variable Importance: INV_MEAN_MIN_DEPTH:
    1.         "Title"  0.739661 ################
    2.           "Age"  0.271536 ##
    3. "FarePerPerson"  0.240145 #
    4.          "Fare"  0.222225 #
    5.          "Deck"  0.206450 #
    6.    "FamilySize"  0.204481 
    7.          "Name"  0.189644 
    8. "Ticket_number"  0.174218 
    9.        "Pclass"  0.171407 
   10.         "Parch"  0.170454 
   11.           "Sex"  0.170310 

Variable Importance: NUM_AS_ROOT:
    1. "Title" 37.000000 ###

In [9]:
model_with_importance = tfdf.keras.GradientBoostedTreesModel(
    verbose=0,
    features=[tfdf.keras.FeatureUsage(name=n) for n in input_features],
    exclude_non_specified_features=True,
    compute_permutation_variable_importance=True,
    random_seed=1234,
)
model_with_importance.fit(train_ds)
model_with_importance.make_inspector().variable_importances()

[INFO 2026-07-05T05:08:05.179228704+00:00 kernel.cc:1214] Loading model from path /tmp/tmpr0_ollom/model/ with prefix 2f9e5b5de281447e
[INFO 2026-07-05T05:08:05.185577769+00:00 kernel.cc:1046] Use fast generic engine


{'MEAN_DECREASE_IN_AUC_2_VS_OTHERS': [("Title" (4; #11), 0.171904761904762),
  ("Deck" (4; #1), 0.053809523809524085),
  ("Pclass" (1; #7), 0.052380952380952195),
  ("FarePerPerson" (1; #4), 0.04000000000000015),
  ("FamilySize" (1; #2), 0.015238095238095273),
  ("Age" (1; #0), 0.013571428571428346),
  ("Name" (5; #5), 0.010952380952380936),
  ("Fare" (1; #3), 0.005714285714286005),
  ("Ticket_number" (4; #10), 0.005714285714285672),
  ("Sex" (4; #8), 0.00547619047619019),
  ("Parch" (1; #6), 0.0004761904761904079),
  ("SibSp" (1; #9), 0.0)],
 'NUM_NODES': [("Name" (5; #5), 174.0),
  ("FarePerPerson" (1; #4), 118.0),
  ("Fare" (1; #3), 86.0),
  ("Age" (1; #0), 78.0),
  ("Deck" (4; #1), 75.0),
  ("Title" (4; #11), 37.0),
  ("FamilySize" (1; #2), 33.0),
  ("Pclass" (1; #7), 26.0),
  ("Ticket_number" (4; #10), 23.0),
  ("Parch" (1; #6), 9.0),
  ("Sex" (4; #8), 2.0),
  ("SibSp" (1; #9), 1.0)],
 'MEAN_DECREASE_IN_AP_2_VS_OTHERS': [("Title" (4; #11), 0.22643784482841578),
  ("Pclass" (1; #7)

# Make predictions

In [10]:
def prediction_to_kaggle_format(model, threshold=0.5):
    proba_survive = model.predict(serving_ds, verbose=0)[:,0]
    return pd.DataFrame({
        "PassengerId": serving_df["PassengerId"],
        "Survived": (proba_survive >= threshold).astype(int)
    })

def make_submission(kaggle_predictions):
    path="/kaggle/working/submission.csv"
    kaggle_predictions.to_csv(path, index=False)
    print(f"Submission exported to {path}")
    
kaggle_predictions = prediction_to_kaggle_format(model)
make_submission(kaggle_predictions)
!head /kaggle/working/submission.csv

Submission exported to /kaggle/working/submission.csv
PassengerId,Survived
892,0
893,0
894,0
895,0
896,0
897,0
898,0
899,0
900,1


# Training a model with hyperparameter tunning

Hyper-parameter tuning is enabled by specifying the tuner constructor argument of the model. The tuner object contains all the configuration of the tuner (search space, optimizer, trial and objective).


In [11]:
tuner = tfdf.tuner.RandomSearch(num_trials=1000)
tuner.choice("min_examples", [2, 5, 7, 10])
tuner.choice("categorical_algorithm", ["CART", "RANDOM"])

local_search_space = tuner.choice("growing_strategy", ["LOCAL"])
local_search_space.choice("max_depth", [3, 4, 5, 6, 8])

global_search_space = tuner.choice("growing_strategy", ["BEST_FIRST_GLOBAL"], merge=True)
global_search_space.choice("max_num_nodes", [16, 32, 64, 128, 256])

#tuner.choice("use_hessian_gain", [True, False])
tuner.choice("shrinkage", [0.02, 0.05, 0.10, 0.15])
tuner.choice("num_candidate_attributes_ratio", [0.2, 0.5, 0.9, 1.0])


tuner.choice("split_axis", ["AXIS_ALIGNED"])
oblique_space = tuner.choice("split_axis", ["SPARSE_OBLIQUE"], merge=True)
oblique_space.choice("sparse_oblique_normalization",
                     ["NONE", "STANDARD_DEVIATION", "MIN_MAX"])
oblique_space.choice("sparse_oblique_weights", ["BINARY", "CONTINUOUS"])
oblique_space.choice("sparse_oblique_num_projections_exponent", [1.0, 1.5])

# Tune the model. Notice the `tuner=tuner`.
tuned_model = tfdf.keras.GradientBoostedTreesModel(tuner=tuner)
tuned_model.fit(train_ds, verbose=0)

tuned_self_evaluation = tuned_model.make_inspector().evaluation()
print(f"Accuracy: {tuned_self_evaluation.accuracy} Loss:{tuned_self_evaluation.loss}")

Use /tmp/tmpnq4f0qjr as temporary training directory


[INFO 2026-07-05T05:10:45.86820243+00:00 kernel.cc:1214] Loading model from path /tmp/tmpnq4f0qjr/model/ with prefix 63d6da88fd9e4b96
[INFO 2026-07-05T05:10:45.885235842+00:00 abstract_model.cc:1311] Engine "GradientBoostedTreesQuickScorerExtended" built
[INFO 2026-07-05T05:10:45.885300854+00:00 kernel.cc:1046] Use fast generic engine


Accuracy: 0.8904109597206116 Loss:0.6197583675384521


In the last line in the cell above, you can see the accuracy is higher than previously with default parameters and parameters set by hand.

This is the main idea behing hyperparameter tuning.

For more information you can follow this tutorial: [Automated hyper-parameter tuning](https://www.tensorflow.org/decision_forests/tutorials/automatic_tuning_colab)

# Making an ensemble

Here you'll create 100 models with different seeds and combine their results

This approach removes a little bit the random aspects related to creating ML models

In the GBT creation is used the `honest` parameter. It will use different training examples to infer the structure and the leaf values. This regularization technique trades examples for bias estimates.

In [12]:
predictions = None
num_predictions = 0

for i in range(100):
    print(f"i:{i}")
    # Possible models: GradientBoostedTreesModel or RandomForestModel
    model = tfdf.keras.GradientBoostedTreesModel(
        verbose=0, # Very few logs
        features=[tfdf.keras.FeatureUsage(name=n) for n in input_features],
        exclude_non_specified_features=True, # Only use the features in "features"

        #min_examples=1,
        #categorical_algorithm="RANDOM",
        ##max_depth=4,
        #shrinkage=0.05,
        ##num_candidate_attributes_ratio=0.2,
        #split_axis="SPARSE_OBLIQUE",
        #sparse_oblique_normalization="MIN_MAX",
        #sparse_oblique_num_projections_exponent=2.0,
        #num_trees=2000,
        ##validation_ratio=0.0,
        random_seed=i,
        honest=True,
    )
    model.fit(train_ds)
    
    sub_predictions = model.predict(serving_ds, verbose=0)[:,0]
    if predictions is None:
        predictions = sub_predictions
    else:
        predictions += sub_predictions
    num_predictions += 1

predictions/=num_predictions

kaggle_predictions = pd.DataFrame({
        "PassengerId": serving_df["PassengerId"],
        "Survived": (predictions >= 0.5).astype(int)
    })

make_submission(kaggle_predictions)

i:0


[INFO 2026-07-05T05:10:47.138393872+00:00 kernel.cc:1214] Loading model from path /tmp/tmpblwt0x6v/model/ with prefix eccee811c0ff4b56
[INFO 2026-07-05T05:10:47.152229769+00:00 kernel.cc:1046] Use fast generic engine


i:1


[INFO 2026-07-05T05:10:48.385291941+00:00 kernel.cc:1214] Loading model from path /tmp/tmpypwfgtoz/model/ with prefix b30cbe2ebbb44f49
[INFO 2026-07-05T05:10:48.395264289+00:00 kernel.cc:1046] Use fast generic engine


i:2


...

i:98


[INFO 2026-07-05T05:12:50.767490695+00:00 kernel.cc:1214] Loading model from path /tmp/tmp5z5mhp5q/model/ with prefix 57afecbf7cdf4949
[INFO 2026-07-05T05:12:50.776973918+00:00 kernel.cc:1046] Use fast generic engine


i:99


[INFO 2026-07-05T05:12:51.881452938+00:00 kernel.cc:1214] Loading model from path /tmp/tmph4x_nr6x/model/ with prefix cd521453dfd64ef5
[INFO 2026-07-05T05:12:51.889386988+00:00 kernel.cc:1046] Use fast generic engine


Submission exported to /kaggle/working/submission.csv
